In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/14 20:05:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
employees = spark.read\
.option("header","true")\
.option("inferSchema", "true")\
.format("csv")\
.load("files/users.txt")

In [4]:
sales = spark.read\
.option("header","true")\
.option("inferSchema", "true")\
.format("csv")\
.load("files/sales.txt")

In [5]:
employees.createOrReplaceTempView("employees")

In [6]:
sales.createOrReplaceTempView("sales")

In [7]:
employees.show()

+---+---------------+-------+----------+----------+
| id|           name| region|  start_dt|    end_dt|
+---+---------------+-------+----------+----------+
|  1|  Alice Johnson|  South|2020-05-15|      null|
|  2|      Bob Smith|  North|2021-02-10|2023-11-30|
|  3|  Charlie Davis|   East|2022-08-22|      null|
|  4|   Diana Prince|   West|2020-01-05|      null|
|  5|  Edward Norton|Central|2021-12-14|      null|
|  6|Fiona Gallagher|  North|2023-03-01|      null|
|  7|  George Miller|  South|2020-11-20|      null|
|  8|  Hannah Abbott|   West|2022-05-18|2024-01-15|
|  9|     Ian Wright|   East|2021-07-09|      null|
| 10|   Jenna Ortega|Central|2020-09-30|      null|
+---+---------------+-------+----------+----------+



In [8]:
sales.show()

+-------+-------+------+--------+-----------+-------------------+
|sale_id|cust_id|emp_id|sale_amt|sale_region|     sale_timestamp|
+-------+-------+------+--------+-----------+-------------------+
|   1001|   5042|     1|  1250.5|      South|2023-01-12 14:30:05|
|   1002|   5120|     3|   450.0|       East|2023-01-14 09:15:22|
|   1003|   5088|     5| 2100.75|    Central|2023-01-15 16:45:10|
|   1004|   5021|     2|   89.99|      North|2023-01-18 11:20:00|
|   1005|   5501|    10|  3400.0|       West|2023-01-20 13:05:45|
|   1006|   5233|     7|  120.25|      South|2023-01-22 10:10:30|
|   1007|   5111|     4|   980.0|       West|2023-01-25 15:55:12|
|   1008|   5092|     9|   150.0|       East|2023-01-28 08:40:00|
|   1009|   5300|     6|  2750.6|      North|2023-02-01 17:20:15|
|   1010|   5155|     8|   620.0|       West|2023-02-03 12:00:00|
+-------+-------+------+--------+-----------+-------------------+



In [15]:
spark.sql(
    """
    select e.id, sum(s.sale_amt) as total_sales
    from employees e join sales s on e.id=s.emp_id and e.region!=s.sale_region
    group by 1
    """
).show()

+---+-----------+
| id|total_sales|
+---+-----------+
| 10|     3400.0|
+---+-----------+



In [18]:
spark.sql(
    """
    select cust_id, emp_id, concat(month(sale_timestamp), '-', year(sale_timestamp)) as month_year , 
    max(sale_amt) as sales
    from sales
    group by 1,2,3
    """
).show()

+-------+------+----------+-------+
|cust_id|emp_id|month_year|  sales|
+-------+------+----------+-------+
|   5111|     4|    1-2023|  980.0|
|   5021|     2|    1-2023|  89.99|
|   5088|     5|    1-2023|2100.75|
|   5501|    10|    1-2023| 3400.0|
|   5120|     3|    1-2023|  450.0|
|   5300|     6|    2-2023| 2750.6|
|   5092|     9|    1-2023|  150.0|
|   5155|     8|    2-2023|  620.0|
|   5233|     7|    1-2023| 120.25|
|   5042|     1|    1-2023| 1250.5|
+-------+------+----------+-------+

